In [120]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [121]:
load_dotenv()

True

In [132]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

In [133]:
class Evalute_essay(BaseModel):

    feedback: str = Field(description="Detailed Feedback for the essay")

    score: float = Field(description="Score the essay out of 10", ge=0, le=10)

In [134]:
structured_model = model.with_structured_output(Evalute_essay)

In [135]:
class Upscstate(TypedDict):
    essay: str
    lf: str
    af: str
    cf: str
    feedback: str
    scores: Annotated[list[float], operator.add]
    avg_score: float

In [136]:
def language(state: Upscstate):
    prompt = (
        "Evaluate the language quality of the following essay and provide feedback "
        "and assign a score out of 10\n"
        f"{state['essay']}"
    )


    output = structured_model.invoke(prompt)

    return {'lf' : output.feedback, 'scores' : [output.score]}

In [137]:
def analysis(state: Upscstate):
    
    prompt = f"Evaluate the depth analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"

    output = structured_model.invoke(prompt)

    return {'af': output.feedback, 'scores': [output.score]}

In [138]:
def clearity(state: Upscstate):
    prompt = (
        "Evaluate the clarity of thought in the following essay and provide feedback "
        "and assign a score out of 10\n"
        f"{state['essay']}"
    )


    output = structured_model.invoke(prompt)

    return {'cf': output.feedback, 'scores': [output.score]}

In [142]:
def final(state: Upscstate):
    prompt = (
        "Based on the following feedbacks create a summarized feedback:\n"
        f"language feedback - {state['lf']}\n"
        f"depth of analysis feedback - {state['af']}\n"
        f"clarity of thought feedback - {state['cf']}"
    )

    feedback = model.invoke(prompt).content

    avgscore = sum(state["scores"])/len(state["scores"])

    return {"feedback": feedback, "avg_score":avgscore}

In [143]:
graph = StateGraph(Upscstate)

graph.add_node("language", language)
graph.add_node("analysis", analysis)
graph.add_node("clearity", clearity)
graph.add_node("final", final)

graph.add_edge(START, 'language')
graph.add_edge(START, 'analysis')
graph.add_edge(START, 'clearity')
graph.add_edge('language', 'final')
graph.add_edge('analysis', 'final')
graph.add_edge('clearity', 'final')
graph.add_edge('final', END)

workflow = graph.compile()


In [ ]:
initial_state = {
    "essay" : '''
        # Artificial Intelligence and Its Impact on Our Life

        Artificial Intelligence is one of the most important technology in todays world. It is also known as AI. Nowadays AI is used in many places like schools, hospitals, offices and even in our mobile phones. It makes our work easy and save lots of time. But there is also some disadvantages of using AI too much.

        First of all, AI is very helpful for students. Students can use AI for doing homework, making notes and understanding difficult topics. If someone have doubt in any subject, they can ask AI and get answers quickly. But many students are using it for copying answers without understanding anything. This is a bad habit because students will not learn properly and their thinking ability can become weak.

        AI is also useful in hospitals and industries. Doctors can take help from AI to find diseases and give better treatment to patients. In factories, machines can do work faster than humans and it also reduce mistakes. Many companies are using AI to complete their work in less time. It also helps people to get information easily from internet.

        However, AI also have some disadvantages. One of the biggest problem is that many people can lose their jobs because machines can do their work. For example, some works which are done by humans can now be done by computers. This can create unemployment in future. Another problem is that AI sometimes gives wrong information but people believe it without checking. So we should not trust AI blindly.

        In my opinion, AI is a very useful technology but we should use it in a proper way. We should not depend on it for everything. Students should use their own brain and learn things instead of just copying from AI. Government should also make some rules so that AI cannot be used for wrong purposes.

        In conclusion, Artificial Intelligence is changing our life in many ways. It have both advantages and disadvantages. If we use it carefully, it can help us in making our future better. But if we use it in wrong way, it can create many problems for society. So it is important to use AI wisely.
    
    '''    
}

workflow.invoke(initial_state)


{'essay': '\n        # Artificial Intelligence and Its Impact on Our Life\n\n        Artificial Intelligence is one of the most important technology in todays world. It is also known as AI. Nowadays AI is used in many places like schools, hospitals, offices and even in our mobile phones. It makes our work easy and save lots of time. But there is also some disadvantages of using AI too much.\n\n        First of all, AI is very helpful for students. Students can use AI for doing homework, making notes and understanding difficult topics. If someone have doubt in any subject, they can ask AI and get answers quickly. But many students are using it for copying answers without understanding anything. This is a bad habit because students will not learn properly and their thinking ability can become weak.\n\n        AI is also useful in hospitals and industries. Doctors can take help from AI to find diseases and give better treatment to patients. In factories, machines can do work faster than h

: 